In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [2]:
# load datasets
finance_df = pd.read_csv("./raw/Personal_Finance_Dataset.csv")
transactions_df = pd.read_csv("./raw/aug_personal_transactions_with_UserId.csv")

In [3]:
finance_df.info

<bound method DataFrame.info of             Date                   Transaction Description       Category  \
0     2020-01-02                               Score each.   Food & Drink   
1     2020-01-02                       Quality throughout.      Utilities   
2     2020-01-04        Instead ahead despite measure ago.           Rent   
3     2020-01-05  Information last everything thank serve.     Investment   
4     2020-01-13              Future choice whatever from.   Food & Drink   
...          ...                                       ...            ...   
1495  2024-12-28                            Quite as when.           Rent   
1496  2024-12-28                   Right analysis mention.  Entertainment   
1497  2024-12-28                    No couple debate must.     Investment   
1498  2024-12-29                  Discussion black follow.       Shopping   
1499  2024-12-29         Pressure activity defense detail.          Other   

       Amount     Type  
0     1485.69  Exp

In [4]:
transactions_df.info

<bound method DataFrame.info of        User ID        Date          Description   Amount Transaction Type  \
0          2.0    1/1/2018               Amazon    11.11            debit   
1          1.0    1/2/2018     Mortgage Payment  1247.44            debit   
2          3.0    1/2/2018      Thai Restaurant    24.22            debit   
3          2.0    1/3/2018  Credit Card Payment  2298.09           credit   
4          2.0    1/4/2018              Netflix    11.76            debit   
...        ...         ...                  ...      ...              ...   
10801      NaN  04/13/2018   Italian Restaurant   124.31           credit   
10802      NaN  01/25/2019      American Tavern   116.59            debit   
10803      NaN  02/09/2020       Hardware Store   183.01           credit   
10804      NaN  05/24/2018   City Water Charges   124.90            debit   
10805      NaN  09/02/2018       Hardware Store    91.19            debit   

                  Category   Account Name  

In [9]:
# no missing values
finance_df.isna().any().any()

False

In [11]:
# missing User IDs
transactions_df.isna().any()

User ID              True
Date                False
Description         False
Amount              False
Transaction Type    False
Category            False
Account Name        False
dtype: bool

In [13]:
(transactions_df.isna()['User ID']).sum()

10000

In [15]:
# no duplicate values
finance_df.duplicated().any()

False

In [17]:
# no duplicate values
transactions_df.duplicated().any()

False

# Data Overview

## Finance Dataset
- Categorical Columns: Transaction Description, Category, Type
- Numerical Columns: Date, Amount

## Transactions Dataset
- Categorical Columns: Description, Transaction Type, Category, Account Name
- Numerical Columns: User ID, Date, Amount

### Issues/Ideas:
- Finance dataset `Transaction Description` column doesn't seem to make sense - drop dataset
- `Date` column should be reformatted to datetime for easier handling - could also create features like day of week, month, etc.
- Transactions dataset has 10000 null values in `User ID` column (Drop column)
- `Category` column (target variable) will need label encoding
- `Description` column will need one-hot encoding (or frequency encoding if this makes the model too slow)
     - May need to group less frequent descriptions into an "Other" group to reduce overfitting on less frequent categories
- `Amount` column needs standardizing (use StandardScaler)
- `Transaction Type` column will need binary label encoding
- `Account Name` column will need one-hot encoding

## Null value handling and datetime formatting

In [21]:
# drop User ID
transactions_df = transactions_df.drop(columns=["User ID"])

In [23]:
# reformat Date from M/D/YYYY strings to datetime
transactions_df["Date"] = pd.to_datetime(transactions_df["Date"], format="%m/%d/%Y", errors="coerce")

In [25]:
# feature engineering from Date
transactions_df["Day of Week"] = transactions_df["Date"].dt.dayofweek
transactions_df["Month"] = transactions_df["Date"].dt.month

In [27]:
transactions_df.head()

,Date,Description,Amount,Transaction Type,Category,Account Name,Day of Week,Month
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card,0,1
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking,1,1
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card,1,1
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2,1
4,2018-01-04,Netflix,11.76,debit,Movies & DVDs,Platinum Card,3,1


## Categorical column encoding

In [30]:
# binary label encode Transaction Type (credit=0, debit=1)
transactions_df["Transaction Type"] = (
    transactions_df["Transaction Type"]
    .str.lower()
    .map({"credit": 0, "debit": 1})
)

In [32]:
# one-hot encode Account Name (only 3 values)
acct_ohe = pd.get_dummies(transactions_df["Account Name"], prefix="Acct", dtype=int)
transactions_df = pd.concat([transactions_df.drop(columns=["Account Name"]), acct_ohe], axis=1)

In [34]:
# label encode Category (target)
transactions_df["Category"] = transactions_df["Category"].astype("category")
category_map = dict(enumerate(transactions_df["Category"].cat.categories))
transactions_df["Category_label"] = transactions_df["Category"].cat.codes

In [36]:

transactions_df[["Category", "Category_label"]].drop_duplicates().sort_values("Category_label").head(10)

,Category,Category_label
48,Alcohol & Bars,0
134,Auto Insurance,1
27,Coffee Shops,2
3,Credit Card Payment,3
565,Electronics & Software,4
192,Entertainment,5
14,Fast Food,6
384,Food & Dining,7
11,Gas & Fuel,8
12,Groceries,9


In [38]:
# standardize Amount
scaler = StandardScaler()
transactions_df["Amount_scaled"] = scaler.fit_transform(transactions_df[["Amount"]])

In [40]:
transactions_df.head()

,Date,Description,Amount,Transaction Type,Category,Day of Week,Month,Acct_Checking,Acct_Platinum Card,Acct_Silver Card,Category_label,Amount_scaled
0,2018-01-01,Amazon,11.11,1,Shopping,0,1,0,1,0,19,-0.520732
1,2018-01-02,Mortgage Payment,1247.44,1,Mortgage & Rent,1,1,1,0,0,14,5.791472
2,2018-01-02,Thai Restaurant,24.22,1,Restaurants,1,1,0,0,1,18,-0.453797
3,2018-01-03,Credit Card Payment,2298.09,0,Credit Card Payment,2,1,0,1,0,3,11.155669
4,2018-01-04,Netflix,11.76,1,Movies & DVDs,3,1,0,1,0,15,-0.517413


In [42]:
out_path = Path("./cleaned/transactions_cleaned_desc.csv")
transactions_df.to_csv(out_path, index=False)